# Main Colab: Bangla hate speech detection with transformer ensembles

Runs the whole thesis pipeline from the GitHub repository. Use **Runtime > Change runtime type > T4 GPU**.

* Every job is **resumable** (finished runs are skipped) and **independent**, so you can run different jobs in different Colab accounts and merge the predictions at the end (Section 6).
* Section order: 1 setup, 2 data, 3 helper, 4 classical baselines, 5 transformers (Study 1/2/3), 6 merge, 7 analysis.
* The own dataset is private: upload it yourself (Section 2). Without it you can still run Study 1 (BD-SHS only) and Section 7 on the released predictions.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/abdullah-mohammed0000/bangla-hate-speech-ensembles.git"
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
else:
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
%cd /content/repo
!pip -q install -r requirements.txt -r requirements-train.txt
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - change the runtime to T4 GPU for Section 5")

## 2. Data
Choose **one** option.

* **Option A (recommended):** upload the four raw zips (`archive__3_.zip` = BD-SHS, `Banhate.zip`, `BanglaHateBert-main.zip`, `Own Data.zip`). The script rebuilds the splits and prints **OK** when they are identical to the thesis files.
* **Option B:** upload `merged_splits.zip` (made earlier by Option A in another account); the files are hash-checked when training starts.

In [ ]:
DATA_MODE = "A"        # "A" = raw zips, "B" = merged_splits.zip
from google.colab import files
import shutil, zipfile
os.makedirs("data/raw", exist_ok=True); os.makedirs("data/splits", exist_ok=True)
up = files.upload()
for name, content in up.items():
    dst = ("data/raw/" if DATA_MODE == "A" else "data/") + name
    open(dst, "wb").write(content)
if DATA_MODE == "A":
    !python scripts/prepare_data.py --raw data/raw --out data/splits
else:
    with zipfile.ZipFile("data/merged_splits.zip") as z: z.extractall("data/splits_tmp")
    import glob
    for f in glob.glob("data/splits_tmp/**/*.csv", recursive=True): shutil.copy(f, "data/splits/")
    from bhs_check import *  # noqa  (no-op; hashes are verified by load_splits in each job)

## 3. Helpers: save progress / merge results

In [ ]:
import glob, zipfile, time
from google.colab import files
sys.path.insert(0, "/content/repo/src")
def save_progress(tag="progress"):
    """Zip all predictions + logs of this session and download it (keep these zips: they are the results)."""
    zp = f"/content/predictions_{tag}_{time.strftime('%Y%m%d_%H%M')}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob("runs/predictions/*"):
            if os.path.isfile(f): z.write(f, os.path.basename(f))
    print("saved", zp); files.download(zp)
def merge_zips():
    """Upload prediction zips from other Colab accounts; their files are copied into runs/predictions."""
    os.makedirs("runs/predictions", exist_ok=True)
    for name, content in files.upload().items():
        with zipfile.ZipFile(io.BytesIO(content)) as z: z.extractall("runs/_merge_tmp")
    n = 0
    for f in glob.glob("runs/_merge_tmp/**/*", recursive=True):
        if os.path.isfile(f):
            dst = "runs/predictions/" + os.path.basename(f)
            if not os.path.exists(dst): shutil.copy(f, dst); n += 1
    shutil.rmtree("runs/_merge_tmp", ignore_errors=True); print("copied", n, "new files")
import io

## 4. Classical baselines (SVM, BiLSTM) - Studies 1 and 2

In [ ]:
!python scripts/train_classical.py --regime A B

## 5. Transformers
Seeds 42, 43, 44 for every job. Suggested split across accounts (each job group is independent):

| Group | Models | Regimes | Study |
|---|---|---|---|
| 5a | banglabert_norm, xlmr | A B | 1, 2 |
| 5b | banglabert, indicbertv2, mbert | A B | 1, 2 |
| 5c | banglabert_large_norm | A B | 1, 2 (the large model needs the most GPU time - one seed per account is fine: change `--seed`) |
| 5d | mbert | C | 2 (size-matched control) |
| 5e | banglabert_norm, xlmr, banglabert_large_norm | O | 3 (own dataset only) |

Run **only the group(s) you need in this session**, then `save_progress()`.

In [ ]:
# 5a - BanglaBERT + normalizer and XLM-R, Single and Combine
!python scripts/train_transformer.py --model banglabert_norm xlmr --regime A B --seed 42 43 44
save_progress("5a")

In [ ]:
# 5b - BanglaBERT, IndicBERTv2, mBERT
!python scripts/train_transformer.py --model banglabert indicbertv2 mbert --regime A B --seed 42 43 44
save_progress("5b")

In [ ]:
# 5c - BanglaBERT-large + normalizer (change --seed to split over sessions, e.g. --seed 42)
!python scripts/train_transformer.py --model banglabert_large_norm --regime A B --seed 42 43 44
save_progress("5c")

In [ ]:
# 5d - size-matched control (mBERT, BD-SHS resampled to the size of the merged set)
!python scripts/train_transformer.py --model mbert --regime C --seed 42 43 44
save_progress("5d")

In [ ]:
# 5e - Study 3: own dataset only
!python scripts/train_transformer.py --model banglabert_norm xlmr banglabert_large_norm --regime O --seed 42 43 44
save_progress("5e")

In [ ]:
# Study 1 only, with just the public BD-SHS files (no other corpus needed)
# !python scripts/train_transformer.py --model banglabert_norm --seed 42 43 44 --bdshs-only data/raw

## 6. Merge predictions from other Colab accounts (optional)

In [ ]:
merge_zips()    # upload the predictions_*.zip files downloaded in the other accounts
print(len(glob.glob("runs/predictions/probs_*.npy")), "prediction files")

## 7. Analysis (CPU, ~1 minute)
Needs the test labels in the prediction folder; they are copied from the data splits below.

In [ ]:
import numpy as np, pandas as pd
from bhs import config as C
from bhs.data import load_splits
D = load_splits("data/splits")
for t in C.TEST_SETS: np.save(f"runs/predictions/labels_{t}.npy", D[t].label.values)
!python scripts/analyze.py --predictions runs/predictions --out results/tables_rerun

In [ ]:
# compare with the thesis numbers (small differences are expected after re-training; see docs/REPRODUCING.md)
!python scripts/verify_release.py --predictions runs/predictions

In [ ]:
# download the tables
!zip -qr tables_rerun.zip results/tables_rerun
files.download("tables_rerun.zip")